# Setup
| Component | Configuration |
|---|---|
| Base model | `distilbert/distilroberta-base` |
| Dataset | `fancyzhx/ag_news` |
| Model class | `AutoModelForSequenceClassification` |
| Trainer | Transformers `Trainer` |
| LoRA task | `TaskType.SEQ_CLS` |
| LoRA target layers | `query`, `value` |
| Additional trainable module | `classifier` |
| Evaluation | Accuracy and macro F1 |

In [1]:
%pip install -q -U transformers datasets peft accelerate scikit-learn google-cloud-bigquery-storage pyarrow==23.0.1 

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
import subprocess
import sys


subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)

check_code = """
import importlib.util
from importlib.metadata import PackageNotFoundError, version

try:
    print("Installed torchao version:", version("torchao"))
except PackageNotFoundError:
    print("torchao package metadata: absent")

print("torchao import location:", importlib.util.find_spec("torchao"))
"""

result = subprocess.run(
    [sys.executable, "-c", check_code],
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

torchao package metadata: absent
torchao import location: None




In [4]:
import os

# Use one GPU, including when Kaggle provides two T4 GPUs.
# Run this before importing torch in a fresh session.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path

import numpy as np
import torch
from kaggle_secrets import UserSecretsClient
from transformers import set_seed


HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

MODEL_ID = "distilbert/distilroberta-base"
DATASET_ID = "fancyzhx/ag_news"

OUTPUT_DIR = Path("/kaggle/working/distilroberta-agnews")
ADAPTER_DIR = OUTPUT_DIR / "final-adapter"

SEED = 42
MAX_LENGTH = 256
SMOKE_TEST = False

set_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings.")

USE_BF16 = torch.cuda.is_bf16_supported()

print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", USE_BF16)
print("Hugging Face token loaded.")

GPU: Tesla T4
BF16 supported: True
Hugging Face token loaded.


In [5]:
from datasets import DatasetDict, load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding


raw = load_dataset(
    DATASET_ID,
    token=HF_TOKEN,
)

label_names = raw["train"].features["label"].names

id2label = {
    index: name
    for index, name in enumerate(label_names)
}
label2id = {
    name: index
    for index, name in id2label.items()
}

# Create validation data from the official training split.
split = raw["train"].train_test_split(
    test_size=0.1,
    stratify_by_column="label",
    seed=SEED,
)

datasets = DatasetDict(
    {
        "train": split["train"],
        "validation": split["test"],
        "test": raw["test"],
    }
)

if SMOKE_TEST:
    datasets["train"] = (
        datasets["train"]
        .shuffle(seed=SEED)
        .select(range(512))
    )
    datasets["validation"] = (
        datasets["validation"]
        .shuffle(seed=SEED)
        .select(range(256))
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized = datasets.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
)

tokenized = tokenized.rename_column("label", "labels")

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

print("Labels:", id2label)
print(tokenized)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

... (6 outputs left out)

Map:   0%|          | 0/108000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Labels: {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 108000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 12000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 7600
    })
})


In [6]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSequenceClassification


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["query", "value"],
    modules_to_save=["classifier"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 741,124 || all params: 82,862,600 || trainable%: 0.8944


In [7]:
import math

from transformers import TrainingArguments


NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4

# Calculate optimizer steps for the single-GPU setup.
batches_per_epoch = math.ceil(
    len(tokenized["train"]) / TRAIN_BATCH_SIZE
)
updates_per_epoch = math.ceil(
    batches_per_epoch / GRADIENT_ACCUMULATION_STEPS
)

total_steps = (
    10
    if SMOKE_TEST
    else updates_per_epoch * NUM_EPOCHS
)

# Warm up over approximately 6% of optimizer steps.
warmup_steps = math.ceil(total_steps * 0.06)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    max_steps=10 if SMOKE_TEST else -1,
    learning_rate=2e-4,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    bf16=USE_BF16,
    fp16=not USE_BF16,
    logging_steps=1 if SMOKE_TEST else 50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    push_to_hub=False,
    label_names=["labels"],
    seed=SEED,
    data_seed=SEED,
)

print("Total optimizer steps:", total_steps)
print("Warm-up steps:", warmup_steps)

Total optimizer steps: 10125
Warm-up steps: 608


In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer


def compute_metrics(eval_prediction):
    logits = eval_prediction.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)
    labels = eval_prediction.label_ids

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.906580,0.195430,0.932250,0.932157
2,0.732960,0.189234,0.934667,0.934678
3,0.931186,0.179072,0.937667,0.937680


In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report


RESULTS_DIR = Path("/kaggle/working/output/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save the trained adapter, classifier, and tokenizer.
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.model.config.save_pretrained(str(ADAPTER_DIR))

if SMOKE_TEST:
    print("Smoke test completed. Official test evaluation skipped.")
else:
    result = trainer.predict(
        tokenized["test"],
        metric_key_prefix="test",
    )

    logits = result.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    predicted_ids = np.argmax(logits, axis=-1)
    true_ids = result.label_ids

    probabilities = torch.softmax(
        torch.as_tensor(logits, dtype=torch.float32),
        dim=-1,
    ).numpy()

    # Overall evaluation metrics: one row.
    metrics_df = pd.DataFrame([result.metrics])
    metrics_df.to_csv(
        RESULTS_DIR / "metrics.csv",
        index=False,
    )

    # Precision, recall, F1, and support for each class.
    report = classification_report(
        true_ids,
        predicted_ids,
        labels=list(range(len(label_names))),
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )

    report_rows = []

    for category, scores in report.items():
        if isinstance(scores, dict):
            report_rows.append(
                {
                    "category": category,
                    **scores,
                }
            )

    report_df = pd.DataFrame(report_rows)
    report_df.to_csv(
        RESULTS_DIR / "classification_report.csv",
        index=False,
    )

    # Predictions follow the original test dataset order.
    predictions_df = pd.DataFrame(
        {
            "test_index": np.arange(len(true_ids)),
            "text": datasets["test"]["text"],
            "true_label_id": true_ids,
            "true_label": [id2label[int(i)] for i in true_ids],
            "predicted_label_id": predicted_ids,
            "predicted_label": [
                id2label[int(i)] for i in predicted_ids
            ],
            "correct": true_ids == predicted_ids,
        }
    )

    for class_id, class_name in id2label.items():
        predictions_df[f"probability_{class_name}"] = (
            probabilities[:, class_id]
        )

    predictions_df.to_csv(
        RESULTS_DIR / "predictions.csv",
        index=False,
    )

    display(metrics_df)
    display(report_df)

    print(f"CSV results saved to: {RESULTS_DIR}")

,test_loss,test_accuracy,test_macro_f1,test_runtime,test_samples_per_second,test_steps_per_second
0,0.192954,0.937632,0.937617,30.0006,253.328,15.833


,category,precision,recall,f1-score,support
0,World,0.955176,0.942105,0.948596,1900.0
1,Sports,0.981704,0.988421,0.985051,1900.0
2,Business,0.912253,0.897368,0.904749,1900.0
3,Sci/Tech,0.901749,0.922632,0.912071,1900.0
4,macro avg,0.937720,0.937632,0.937617,7600.0
5,weighted avg,0.937720,0.937632,0.937617,7600.0


CSV results saved to: /kaggle/working/output/results


In [10]:
device = next(trainer.model.parameters()).device


def classify_news(text):
    inputs = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    trainer.model.eval()

    with torch.inference_mode():
        logits = trainer.model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = probabilities.argmax().item()

    return {
        "label": id2label[predicted_id],
        "scores": {
            id2label[index]: float(score)
            for index, score in enumerate(probabilities.cpu().tolist())
        },
    }


classify_news(
    "The team scored twice in the final minutes to win the championship."
)

{'label': 'Sports',
 'scores': {'World': 0.0015667594270780683,
  'Sports': 0.998286783695221,
  'Business': 6.69817891321145e-05,
  'Sci/Tech': 7.954284956213087e-05}}

In [11]:
import shutil

from IPython.display import FileLink, display


archive_path = shutil.make_archive(
    "/kaggle/working/distilroberta-agnews-adapter",
    "zip",
    root_dir=str(ADAPTER_DIR),
)

display(FileLink(archive_path))

/kaggle/working/distilroberta-agnews-adapter.zip